# Verified deep insights: scrap discovery

Run in a Microsoft Fabric Python 3.12 notebook with `semantic-link-sempy` available, semantic-model read/query permissions (including Build where required), and access to the configured Fabric AI model. The semantic model should expose governed scrap-rate measures, their numerator and denominator, dates, plants, and product categories. Names and definitions are discovered per model. No Lakehouse is required. Model availability depends on your workspace and region; runs consume capacity.

Here, **verified** means portable skill structure/arithmetic checks only, not an automatic independent host-side semantic-model source audit or full proof of correctness. The task asks the worker to execute independent source reconciliation; review that evidence. Markdown prose is not verified. Verifier errors may degrade validation rather than block submission, so inspect the trace and verifier diagnostics before relying on findings. The skill requires at least one supported insight; an empty findings list is rejected. If none can be supported, do not manufacture a finding: the run may end without an accepted report. Offline tests do not validate live Fabric authentication or model quality.

In [ ]:
%pip install "fabric-rlm[analytics]==0.6.5"

**Restart the Python session now**, then run from the configuration cell below, not the install cell. Replace the workspace and semantic-model IDs; edit the task literal in the main cell for your discovery question. `SemanticModel` uses its default automatic authentication and `FabricLM` uses your Fabric identity; no API keys are entered here.

In [ ]:
from fabric_rlm import RLM, FabricLM, SemanticModel
from IPython.display import Markdown, display

WORKSPACE_ID = "<workspace-id>"
MODEL_ID = "<semantic-model-id>"
MODEL_NAME = "gpt-5.1"

source = SemanticModel(MODEL_ID, workspace=WORKSPACE_ID)
lm = FabricLM(MODEL_NAME, reasoning_effort="high")

In [ ]:
result = RLM.task(
    task="""Discover non-obvious scrap-rate insights in the bound semantic model.
Inspect governed measures, their definitions, relationships, and available periods
before querying. Use the model's own scrap-rate definition and identify its
numerator, denominator, units, and filter context; do not invent measure names.

Compare the latest two complete periods supported by source coverage, not merely
the latest dates in a calendar table. Check coverage and completeness for both
periods. If completeness cannot be confirmed, disclose that limitation and label
any available-period comparison provisional, not a confirmed current change.
Record the actual periods, coverage evidence, and excluded partial periods.

Explore plant and product category, including their intersection where supported.
Distinguish within-group rate changes from production/product mix changes.
Use denominator-weighted rates, not an average of subgroup percentages. Execute
independent source queries to reconcile numerator, denominator, and aggregate
rates under the same filters. Reconcile within-group and mix contributions to
the observed rate change with an explicit residual. Execute the verification
expressions for submitted claims/components and compare the results before
submission; merely writing expressions is not reconciliation. Disclose gaps
instead of treating missing data as zero or fabricating successful checks.

Follow the deep_insight_discovery skill contract version 2, including typed
diagnostics. Evaluate at most 3 actual candidates and record their dispositions
in the candidate ledger. Defer unsearched dimensions with reasons. Do not invent
candidates or rejected findings to fill a quota; submit between one and three
supported insights. If none can be supported, explain the limitation in execution
feedback and do not submit fabricated insights. Do not invent scrap reasons, causal explanations,
or materiality thresholds. Separate measured facts from hypotheses, assess
competing explanations, and retain investigate-first recommendations where
measurable alternatives remain unresolved.

Also submit report_markdown as a faithful rendering of the structured findings.
Use readable Markdown headings and tables for measured facts, periods,
comparisons, and limitations. Preserve uncertainty and candidate dispositions.
Include no extra numbers or claims beyond the structured outputs, and no code
fences. Preserve any limitations on which findings can be promoted.""",
    inputs={"source": source},
    outputs={
        "contract_version": int,
        "analysis_plan": dict,
        "candidates": list,
        "insights": list,
        "report_markdown": str,
    },
    lm=lm,
    skills=["semantic_model", "analytical_integrity", "deep_insight_discovery"],
    enable_verifier=True,
    max_turns=25,
).run()

In [ ]:
assert result.submitted, result.report()
display(Markdown(result.outputs["report_markdown"]))
result.outputs

In [ ]:
# Optional diagnostics; use result.inspect() to review the full trace.
result.trajectory.metadata.get("verifier_execution")